Try cuda-graph end-to-end model

In [1]:
import torch
from torch import nn
from model import Transformer, ModelArgs, Linear, ParallelEmbedding, Gate

In [2]:
device = "cuda:0"
torch.cuda.set_device(device)

torch.set_default_dtype(torch.bfloat16)
torch.set_num_threads(8)
torch.manual_seed(965)

In [3]:
def report_memory():
    print(
        f"Allocated: {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB, "
        f"Reserved: {torch.cuda.memory_reserved() / 1024 ** 3:.2f} GB, "
        f"Max Allocated: {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB, "
        f"Max Reserved: {torch.cuda.max_memory_reserved() / 1024 ** 3:.2f} GB"
    )

report_memory()

Allocated: 0.00 GB, Reserved: 0.00 GB, Max Allocated: 0.00 GB, Max Reserved: 0.00 GB


In [4]:
# small size for fast on-device test
args = ModelArgs(
    vocab_size=32000,
    dim=512,
    inter_dim=2736,
    moe_inter_dim=352,
    n_layers=3,
    n_dense_layers=1,
    n_routed_experts=16,
    n_shared_experts=2,
    n_activated_experts=6,
)
args

ModelArgs(max_batch_size=8, max_seq_len=16384, dtype='bf16', vocab_size=32000, dim=512, inter_dim=2736, moe_inter_dim=352, n_layers=3, n_dense_layers=1, n_heads=16, n_routed_experts=16, n_shared_experts=2, n_activated_experts=6, n_expert_groups=1, n_limited_groups=1, score_func='softmax', route_scale=1.0, q_lora_rank=0, kv_lora_rank=512, qk_nope_head_dim=128, qk_rope_head_dim=64, v_head_dim=128, original_seq_len=4096, rope_theta=10000.0, rope_factor=40, beta_fast=32, beta_slow=1, mscale=1.0)

In [5]:
with torch.device(device):
    model = Transformer(args)

# model

In [6]:
report_memory()

Allocated: 0.56 GB, Reserved: 0.59 GB, Max Allocated: 0.57 GB, Max Reserved: 0.59 GB


In [7]:
model.layers[0].ffn.w1.weight  # weight not initialized!

Parameter containing:
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0', requires_grad=True)

In [8]:
def init_weights_normal(m):
    if isinstance(m, (Linear, ParallelEmbedding, Gate)):
        nn.init.normal_(m.weight, mean=0.0, std=0.1)

model.apply(init_weights_normal);

In [9]:
model.layers[0].ffn.w1.weight  # initialized to random weights

Parameter containing:
tensor([[ 0.0420, -0.0791,  0.1484,  ...,  0.1025, -0.1299, -0.0125],
        [ 0.0688, -0.0811,  0.1172,  ..., -0.1709,  0.1309, -0.0417],
        [ 0.0986, -0.1172, -0.0291,  ...,  0.2383,  0.0410,  0.0242],
        ...,
        [ 0.0012, -0.1553,  0.0160,  ...,  0.0151,  0.1992,  0.1060],
        [ 0.0591, -0.0544,  0.1035,  ..., -0.0214, -0.1099, -0.0623],
        [ 0.0698,  0.0635, -0.0830,  ..., -0.1338, -0.0041, -0.1309]],
       device='cuda:0', requires_grad=True)

In [10]:
model.embed.weight

Parameter containing:
tensor([[ 0.0150, -0.0072, -0.2500,  ...,  0.1211, -0.1187,  0.0664],
        [-0.1328, -0.1182,  0.0894,  ..., -0.0515, -0.0728, -0.1104],
        [ 0.1050, -0.1094, -0.0962,  ..., -0.0664,  0.1553, -0.0147],
        ...,
        [-0.0698,  0.0889, -0.1367,  ..., -0.0698, -0.0108, -0.1152],
        [-0.0053, -0.1641, -0.1309,  ..., -0.0374, -0.0184, -0.2080],
        [-0.0498,  0.0300, -0.0100,  ...,  0.0038, -0.0190,  0.0334]],
       device='cuda:0', requires_grad=True)

In [11]:
batch = 4
num_tokens = 50
tokens = torch.randint(0, args.vocab_size, size=(batch, num_tokens), dtype=torch.int64).to(device)
output = model.forward(tokens)
print(f"output.shape = {output.shape}")
print(f"output.min = {output.min()}, output.max = {output.max()}")

output.shape = torch.Size([4, 50, 32000])
output.min = -12.0625, output.max = 12.6875


In [12]:
# import matplotlib.pyplot as plt
# plt.hist(output.flatten().to(torch.float32).cpu(), bins=50);  # look quite normal

In [13]:
compiled_model = torch.compile(
    model,
    backend="cudagraphs",
    dynamic=False,
    fullgraph=True
)

In [14]:
# output_compiled = compiled_model.forward(tokens)  # failed due to torch.bincount